# T4 GGUF Qwen2.5 7B Chat on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kingwabg/supergemma4-colab-runner/blob/main/notebooks/T4_GGUF_Qwen2_5_7B_Colab.ipynb)

이 노트북은 무료 Colab T4에서 26B MLX 모델 대신 더 작은 GGUF 모델을 실행하기 위한 경로입니다.

기본값은 `Qwen2.5-7B-Instruct-GGUF`의 `Q4_K_M` 파일입니다. 품질과 속도의 균형이 좋지만, T4 런타임 상태에 따라 느리거나 메모리가 부족할 수 있습니다. 그 경우 설정 셀에서 `MODEL_PRESET = "ultra_light"`로 바꿔 `IQ2_M` 파일을 쓰세요.

주의:

- API 키가 필요 없습니다. 모델 파일은 Hugging Face에서 런타임마다 내려받습니다.
- Colab 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU`를 먼저 선택하세요.
- 무료 Colab GPU 종류와 사용 시간은 보장되지 않습니다.
- 민감한 개인정보나 비밀 업무 데이터를 프롬프트에 넣지 마세요.


## 1. GPU 확인

아래 셀에서 `Tesla T4`가 보이면 정상입니다. GPU가 없으면 `런타임 유형 변경`에서 T4 GPU를 선택한 뒤 다시 실행하세요.


In [ ]:
!nvidia-smi


## 2. llama-cpp-python CUDA 설치

Colab CUDA 12.x 런타임 기준으로 CUDA 휠을 설치합니다. 설치가 실패하면 아래 셀의 `cu124`를 `cu121` 또는 `cu125`로 바꿔 다시 실행해보세요.


In [ ]:
%pip install -q -U huggingface_hub
%pip install -q -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


## 3. 모델 선택 및 다운로드

- `balanced`: Qwen2.5 7B `Q4_K_M`. 기본 추천값입니다.
- `ultra_light`: Qwen2.5 7B `IQ2_M`. 더 가볍지만 답변 품질이 떨어질 수 있습니다.

T4에서 `balanced`가 너무 느리거나 OOM이면 `MODEL_PRESET = "ultra_light"`로 바꾼 뒤 이 셀부터 다시 실행하세요.


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_PRESET = "balanced"  # "balanced" 또는 "ultra_light"

MODELS = {
    "balanced": {
        "repo_id": "lmstudio-community/Qwen2.5-7B-Instruct-GGUF",
        "filename": "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
        "n_ctx": 2048,
        "max_tokens": 256,
    },
    "ultra_light": {
        "repo_id": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "filename": "Qwen2.5-7B-Instruct-IQ2_M.gguf",
        "n_ctx": 1536,
        "max_tokens": 192,
    },
}

config = MODELS[MODEL_PRESET]
print("선택한 모델:", config["repo_id"], config["filename"])

MODEL_PATH = hf_hub_download(
    repo_id=config["repo_id"],
    filename=config["filename"],
    local_dir="/content/models",
)

print("다운로드 완료:", MODEL_PATH)


## 4. 모델 로드

`n_gpu_layers=-1`은 가능한 레이어를 GPU에 올리라는 뜻입니다. 여기서 OOM이 나면 `n_ctx`를 1024로 줄이거나 `MODEL_PRESET`을 `ultra_light`로 바꾸세요.


In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=config["n_ctx"],
    n_batch=512,
    verbose=True,
)

print("모델 로드 완료")


## 5. 단발 생성 테스트

먼저 짧은 답변으로 정상 동작을 확인합니다.


In [ ]:
SYSTEM_PROMPT = "당신은 한국어로 짧고 정확하게 답하는 AI입니다. 모르면 모른다고 말합니다."

def build_qwen_prompt(messages):
    text = ""
    for message in messages:
        role = message["role"]
        content = message["content"].strip()
        text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    text += "<|im_start|>assistant\n"
    return text

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "무료 Colab T4에서 7B GGUF 모델을 쓰는 장점을 한 문장으로 말해줘."},
]

result = llm(
    build_qwen_prompt(messages),
    max_tokens=128,
    temperature=0.7,
    top_p=0.9,
    stop=["<|im_end|>", "<|im_start|>"],
)

print(result["choices"][0]["text"].strip())


## 6. 한 번 질문하기

`모두 실행`해도 멈추지 않도록 기본 질문 문자열을 사용합니다. 원하는 질문으로 `QUESTION` 값만 바꾼 뒤 이 셀을 다시 실행하세요.


In [ ]:
QUESTION = "무료 Colab T4에서 GGUF 모델을 쓰는 장점을 짧게 말해줘."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": QUESTION},
]

result = llm(
    build_qwen_prompt(messages),
    max_tokens=config["max_tokens"],
    temperature=0.7,
    top_p=0.9,
    repeat_penalty=1.08,
    stop=["<|im_end|>", "<|im_start|>"],
)

print("질문:", QUESTION)
print("\nAI:", result["choices"][0]["text"].strip())


## 7. 선택: 연속 채팅

연속으로 대화하고 싶을 때만 아래 셀의 `RUN_CONTINUOUS_CHAT = True`로 바꾸고 따로 실행하세요. 기본값은 `False`라서 `모두 실행`해도 멈추지 않습니다.


In [ ]:
RUN_CONTINUOUS_CHAT = False

if not RUN_CONTINUOUS_CHAT:
    print("연속 채팅은 기본 OFF입니다. 필요하면 RUN_CONTINUOUS_CHAT = True로 바꾸고 이 셀만 다시 실행하세요.")
else:
    history = []

    while True:
        user_text = input("\n나: ").strip()
        if user_text.lower() in {"/exit", "/quit", "exit", "quit"}:
            print("종료합니다.")
            break
        if not user_text:
            continue

        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages.extend(history[-6:])
        messages.append({"role": "user", "content": user_text})

        result = llm(
            build_qwen_prompt(messages),
            max_tokens=config["max_tokens"],
            temperature=0.7,
            top_p=0.9,
            repeat_penalty=1.08,
            stop=["<|im_end|>", "<|im_start|>"],
        )

        answer = result["choices"][0]["text"].strip()
        print("\nAI:", answer)

        history.append({"role": "user", "content": user_text})
        history.append({"role": "assistant", "content": answer})


## 문제가 생기면

- 모델 로드 중 OOM: `MODEL_PRESET = "ultra_light"`로 변경 후 3번 셀부터 다시 실행하세요.
- 답변이 너무 느림: `config["max_tokens"]`를 128 이하로 줄이세요.
- 긴 대화에서 느려짐: `n_ctx`를 1024로 낮추고 런타임을 다시 시작하세요.
- CUDA 휠 설치 실패: 2번 셀의 `cu124`를 `cu121`, `cu125` 중 Colab CUDA 버전에 맞는 값으로 바꿔 재설치하세요.
